# SCD2 Spark and dbt
**Modules 7-8 · Intermediate to Advanced** · part 4 of 8 of *Building Automated Data Pipelines with Spark, dbt and Airflow* (UK · USA · Mauritius job-market aligned)

Each notebook is **self-contained**: run top to bottom. Needs Python 3.9+ and Java 8/11/17 for PySpark (not needed for 01, 05, 08).

**Series:**
- `01_DFD_Data_Flow_Diagrams.ipynb`: Modules 1-2 · Beginner
- `02_Modular_Pipeline.ipynb`: Modules 3-4 · Beginner to Intermediate
- `03_Connectors_and_Integration.ipynb`: Modules 5-6 · Intermediate
- `04_SCD2_Spark_and_dbt.ipynb`: Modules 7-8 · Intermediate to Advanced
- `05_Airflow_Workflow_Design.ipynb`: Modules 9-10 · Advanced
- `06_Capstone_End_to_End.ipynb`: Module 11 · Advanced
- `07_Advanced_Performance_Testing_CICD.ipynb`: Beyond the course · Advanced
- `08_Career_UK_USA_Mauritius.ipynb`: Job-market alignment · All levels

## Setup

In [ ]:
# In Colab / fresh env, uncomment:
# !pip install -q pyspark==3.5.3 pandas numpy matplotlib pyyaml

import os, sys, json, time, shutil, logging, sqlite3, ast
from pathlib import Path
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pyspark.sql import SparkSession, DataFrame, functions as F, Window

spark = (SparkSession.builder.master("local[*]").appName("pipelines-lab")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

BASE = Path("pipeline_lab"); BASE.mkdir(exist_ok=True)
DATA = BASE / "data"; DATA.mkdir(exist_ok=True)
WH   = BASE / "warehouse"
print("Spark", spark.version, "| workspace:", BASE.resolve())

## Lab data (simulated source systems)
We simulate three real-world sources, with *deliberately dirty* data (duplicates, bad amounts, nulls, mixed-case currency, late-arriving customer) so you practise real problems.

In [ ]:
rng = np.random.default_rng(42)

# --- customers: day 1 vs day 2 (changes + new customer) ---
cust_day1 = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6],
    "name":    ["Asha Patel", "Liam Smith", "Chloe Dubois", "Raj Ramgoolam", "Emma Jones", "Noah Brown"],
    "email":   [f"user{i}@example.com" for i in range(1, 7)],
    "city":    ["London", "Manchester", "Port Louis", "Curepipe", "New York", "Austin"],
    "segment": ["gold", "silver", "silver", "bronze", "gold", "bronze"],
})
cust_day2 = cust_day1.copy()
cust_day2.loc[cust_day2.customer_id == 2, "city"] = "Edinburgh"      # customer moved
cust_day2.loc[cust_day2.customer_id == 4, "segment"] = "silver"      # upgraded
cust_day2 = pd.concat([cust_day2, pd.DataFrame([{"customer_id": 7, "name": "Priya Nair", "email": "user7@example.com",
                                                 "city": "Ebene", "segment": "bronze"}])], ignore_index=True)

def simulate_source(day: int):
    con = sqlite3.connect(DATA / "crm.db")
    (cust_day1 if day == 1 else cust_day2).to_sql("customers", con, if_exists="replace", index=False)
    con.close()

# --- orders CSV (dirty on purpose) ---
n = 40
orders = pd.DataFrame({
    "order_id": np.arange(1000, 1000 + n),
    "customer_id": rng.integers(1, 8, n),
    "order_date": (pd.Timestamp("2026-09-01") + pd.to_timedelta(rng.integers(0, 28, n), unit="D")).strftime("%Y-%m-%d"),
    "amount": rng.uniform(5, 300, n).round(2),
    "currency": rng.choice(["GBP", "USD", "MUR"], n),
})
dirty = pd.concat([orders.iloc[[0, 1]],                                                       # duplicates
                   pd.DataFrame([{"order_id": 2001, "customer_id": 3, "order_date": "2026-09-10", "amount": -5.0, "currency": "GBP"},   # negative amount
                                 {"order_id": None, "customer_id": 2, "order_date": "2026-09-11", "amount": 20.0, "currency": "USD"},  # null key
                                 {"order_id": 2003, "customer_id": 1, "order_date": "2026-09-12", "amount": 99.0, "currency": " gbp "}])])  # messy currency
orders = pd.concat([orders, dirty], ignore_index=True)
orders.to_csv(DATA / "orders.csv", index=False)

# --- web events JSON lines ---
events = pd.DataFrame({
    "customer_id": rng.integers(1, 8, 60),
    "event_type": rng.choice(["view", "click", "add_to_cart", "purchase"], 60),
    "ts": (pd.Timestamp("2026-09-01") + pd.to_timedelta(rng.integers(0, 28 * 24 * 3600, 60), unit="s")).strftime("%Y-%m-%dT%H:%M:%S"),
})
events.to_json(DATA / "events.jsonl", orient="records", lines=True)

simulate_source(1)
print("orders rows:", len(orders), "| events:", len(events), "| customers (day1):", len(cust_day1))
orders.tail(6)

## Prerequisites from earlier notebooks (run once; skim if you already did them)

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("pipeline")

def timed(fn):
    def wrapper(*a, **kw):
        t = time.time(); out = fn(*a, **kw)
        log.info("step=%s duration=%.2fs", fn.__name__, time.time() - t)
        return out
    wrapper.__name__ = fn.__name__
    return wrapper

@timed
def clean_orders(df: DataFrame):
    # Standardise types and split into (valid, quarantined)
    typed = (df.withColumn("order_id", F.col("order_id").cast("long"))
               .withColumn("customer_id", F.col("customer_id").cast("int"))
               .withColumn("order_date", F.to_date("order_date"))
               .withColumn("amount", F.col("amount").cast("double"))
               .withColumn("currency", F.upper(F.trim("currency"))))
    is_valid = F.coalesce(F.col("order_id").isNotNull() & (F.col("amount") > 0) & F.col("order_date").isNotNull(), F.lit(False))
    good = typed.filter(is_valid).dropDuplicates(["order_id"])
    bad = typed.filter(~is_valid).withColumn("reject_reason", F.lit("null key / non-positive amount / bad date"))
    return good, bad

raw_orders = spark.read.option("header", True).csv(str(DATA / "orders.csv"))
orders_ok, orders_bad = clean_orders(raw_orders)
print("raw:", raw_orders.count(), "| valid:", orders_ok.count(), "| quarantined:", orders_bad.count())
orders_bad.show(truncate=False)

In [ ]:
# Data-quality framework: (name, check, severity). 'error' blocks the pipeline, 'warn' only logs.
ORDER_CHECKS = [
    ("order_id is unique",       lambda df: df.count() == df.select("order_id").distinct().count(), "error"),
    ("order_id not null",        lambda df: df.filter("order_id IS NULL").count() == 0,              "error"),
    ("amount > 0",               lambda df: df.filter("amount <= 0").count() == 0,                   "error"),
    ("currency in allowed set",  lambda df: df.filter(~F.col("currency").isin("GBP", "USD", "MUR")).count() == 0, "error"),
    ("row count > 0",            lambda df: df.count() > 0,                                          "error"),
]

def run_dq(df, checks):
    results = []
    for name, fn, severity in checks:
        passed = bool(fn(df))
        results.append({"check": name, "passed": passed, "severity": severity})
        (log.info if passed else log.warning)("DQ %-28s %s", name, "PASS" if passed else "FAIL")
    failures = [r for r in results if not r["passed"] and r["severity"] == "error"]
    if failures:
        raise ValueError(f"Data quality gate failed: {failures}")
    return results

dq_results = run_dq(orders_ok, ORDER_CHECKS)
pd.DataFrame(dq_results)

In [ ]:
from abc import ABC, abstractmethod

class Connector(ABC):
    def __init__(self, cfg): self.cfg = cfg
    @abstractmethod
    def read(self, spark) -> DataFrame: ...

class CSVConnector(Connector):
    def read(self, spark):
        return spark.read.options(**{"header": "true", **self.cfg.get("options", {})}).csv(self.cfg["path"])

class JSONConnector(Connector):
    def read(self, spark):
        return spark.read.json(self.cfg["path"])

class SQLiteConnector(Connector):          # stand-in for JDBC: spark.read.format("jdbc").option("url", ...).load()
    def read(self, spark):
        con = sqlite3.connect(self.cfg["path"])
        pdf = pd.read_sql_query(f"SELECT * FROM {self.cfg['table']}", con); con.close()
        return spark.createDataFrame(pdf)

REGISTRY = {"csv": CSVConnector, "json": JSONConnector, "sqlite": SQLiteConnector}
def build_connector(cfg): return REGISTRY[cfg["type"]](cfg)

CONFIG_YAML = f'''
sources:
  crm_customers:
    type: sqlite
    path: {DATA / "crm.db"}
    table: customers
  orders:
    type: csv
    path: {DATA / "orders.csv"}
  web_events:
    type: json
    path: {DATA / "events.jsonl"}
'''
CFG = yaml.safe_load(CONFIG_YAML)
raw = {name: build_connector(c).read(spark) for name, c in CFG["sources"].items()}
for k, v in raw.items():
    print(f"--- {k}: {v.count()} rows"); v.printSchema()

In [ ]:
FX = spark.createDataFrame([("GBP", 1.0), ("USD", 0.78), ("MUR", 0.017)], ["currency", "to_gbp"])   # illustrative rates only

def build_customer_360(customers_current: DataFrame, orders_clean: DataFrame, events_raw: DataFrame) -> DataFrame:
    o = (orders_clean.join(F.broadcast(FX), "currency", "left")
                     .groupBy("customer_id")
                     .agg(F.count("*").alias("orders"),
                          F.round(F.sum(F.col("amount") * F.col("to_gbp")), 2).alias("revenue_gbp"),
                          F.max("order_date").alias("last_order_date")))
    e = events_raw.groupBy("customer_id").agg(F.count("*").alias("web_events"),
                                              F.sum((F.col("event_type") == "purchase").cast("int")).alias("web_purchases"))
    return (customers_current.join(o, "customer_id", "left").join(e, "customer_id", "left")
            .fillna({"orders": 0, "revenue_gbp": 0.0, "web_events": 0, "web_purchases": 0}))

c360 = build_customer_360(raw["crm_customers"], orders_ok, raw["web_events"])
c360.orderBy("customer_id").show()
# Orphan check: orders whose customer does not exist yet (late-arriving dimension)
print("orphan orders:", orders_ok.join(raw["crm_customers"], "customer_id", "left_anti").count())

# PART 4: SCD Type 2 historical tracking (Modules 7–8) · Intermediate → Advanced

**Problem:** Liam moves from Manchester to Edinburgh. If you overwrite the row, last month's orders look like they came from Edinburgh. **SCD2 keeps every version** with validity dates.

| customer_id | city | effective_from | effective_to | is_current |
|---|---|---|---|---|
| 2 | Manchester | 2026-09-01 | 2026-09-02 | false |
| 2 | Edinburgh | 2026-09-02 | 9999-12-31 | true |

We use half-open intervals **[from, to)**. We detect change with a **hash of tracked columns** (compact, fast, null-safe).

### 4A. SCD2 in PySpark

In [ ]:
HIGH_DATE = "9999-12-31"

def add_hash(df, tracked):
    parts = [F.coalesce(F.col(c).cast("string"), F.lit("<NULL>")) for c in tracked]
    return df.withColumn("row_hash", F.sha2(F.concat_ws("||", *parts), 256))

def apply_scd2(dim, incoming, key, tracked, load_date):
    # Returns the new full dimension (history + current versions). dim=None for the initial load.
    cols = [key] + tracked
    inc = add_hash(incoming.select(*cols).dropDuplicates([key]), tracked)
    ld, hd = F.lit(load_date).cast("date"), F.lit(HIGH_DATE).cast("date")

    def as_new_version(df):
        return (df.select(*cols, "row_hash").withColumn("effective_from", ld)
                  .withColumn("effective_to", hd).withColumn("is_current", F.lit(True)))

    if dim is None:
        return as_new_version(inc)

    hist = dim.filter(~F.col("is_current"))
    cur = dim.filter(F.col("is_current"))
    j = inc.alias("i").join(cur.alias("c"), F.col("i." + key) == F.col("c." + key), "full_outer")
    is_new = F.col("c." + key).isNull()
    is_gone = F.col("i." + key).isNull()
    is_changed = (~is_new) & (~is_gone) & (F.col("i.row_hash") != F.col("c.row_hash"))

    keep = j.filter((~is_new) & (~is_changed)).select("c.*")                        # unchanged, or absent from feed (kept open)
    expired = (j.filter(is_changed).select("c.*")
                .withColumn("effective_to", ld).withColumn("is_current", F.lit(False)))
    fresh = as_new_version(j.filter(is_new | is_changed).select("i.*"))           # brand-new keys + new versions of changed keys
    return hist.unionByName(keep).unionByName(expired).unionByName(fresh)

def assert_scd2_ok(dim, key):
    dup_current = dim.filter("is_current").groupBy(key).count().filter("count > 1").count()
    w = Window.partitionBy(key).orderBy("effective_from")
    gaps = (dim.withColumn("next_from", F.lead("effective_from").over(w))
               .filter(F.col("next_from").isNotNull() & (F.col("next_from") != F.col("effective_to"))).count())
    assert dup_current == 0, "more than one current row for a key"
    assert gaps == 0, "gap or overlap in validity ranges"
    return True

TRACKED = ["name", "city", "segment"]
dim_v1 = apply_scd2(None, spark.createDataFrame(cust_day1), "customer_id", TRACKED, "2026-09-01")
dim_v2 = apply_scd2(dim_v1, spark.createDataFrame(cust_day2), "customer_id", TRACKED, "2026-09-02")
print("SCD2 integrity:", assert_scd2_ok(dim_v2, "customer_id"))
dim_v2.orderBy("customer_id", "effective_from").drop("row_hash").show(20, truncate=False)

**Point-in-time join (the reason SCD2 exists):** attach the customer attributes that were valid *on the order date*.

In [ ]:
pit = (orders_ok.alias("o")
       .join(dim_v2.alias("d"),
             (F.col("o.customer_id") == F.col("d.customer_id")) &
             (F.col("o.order_date") >= F.col("d.effective_from")) & (F.col("o.order_date") < F.col("d.effective_to")), "left")
       .select("o.order_id", "o.order_date", "o.customer_id", "d.city", "d.segment", "d.effective_from"))
pit.filter("customer_id = 2").orderBy("order_date").show(8)

Note: with only two simulated loads on 1 and 2 September, almost all orders fall after day 2. In production the dim is loaded daily so history is rich. **Exercise 4.1:** backdate `load_date` for day 1 to `2026-08-01` and re-run to see Manchester orders in the PIT result.

**Edge cases seniors discuss:** hard deletes (we keep rows open; alternatively close them with a `deleted` flag), multiple changes in one day (take latest per key), late-arriving changes (needs reprocessing/backfill), and tracking only *business-relevant* columns to avoid version explosion.

### 4B. SCD2 in dbt (snapshots)
dbt gives you SCD2 declaratively. Below we generate a complete dbt project (DuckDB adapter so it runs without a warehouse). In UK/US teams dbt targets Snowflake, BigQuery, Databricks, Redshift, Postgres or Synapse, so only `profiles.yml` changes.

In [ ]:
DBT = BASE / "dbt_project"
for d in ["models/staging", "models/marts", "snapshots", "seeds", "tests"]:
    (DBT / d).mkdir(parents=True, exist_ok=True)

(DBT / "dbt_project.yml").write_text('''
name: pipeline_lab
version: "1.0.0"
config-version: 2
profile: pipeline_lab
model-paths: ["models"]
seed-paths: ["seeds"]
snapshot-paths: ["snapshots"]
test-paths: ["tests"]
models:
  pipeline_lab:
    staging: {+materialized: view}
    marts:   {+materialized: table}
''')

(DBT / "profiles.yml").write_text('''
pipeline_lab:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: lab.duckdb
      threads: 4
    # prod example: add a second output (snowflake / bigquery / databricks) and run with --target prod
''')

# Seeds = our source data (day-1 customers + orders)
cust_day1.to_csv(DBT / "seeds/customers.csv", index=False)
orders.to_csv(DBT / "seeds/orders.csv", index=False)
cust_day2.to_csv(DATA / "customers_day2.csv", index=False)       # swap into seeds/customers.csv to simulate day 2

(DBT / "snapshots/customers_snapshot.sql").write_text('''
{% snapshot customers_snapshot %}
{{ config(target_schema='snapshots', unique_key='customer_id',
          strategy='check', check_cols=['name', 'city', 'segment']) }}
select * from {{ ref('customers') }}
{% endsnapshot %}
''')

(DBT / "models/staging/stg_orders.sql").write_text('''
with src as (select * from {{ ref('orders') }})
select cast(order_id as bigint) as order_id,
       cast(customer_id as int) as customer_id,
       cast(order_date as date) as order_date,
       cast(amount as double) as amount,
       upper(trim(currency)) as currency
from src
where order_id is not null and amount > 0
qualify row_number() over (partition by order_id order by order_date) = 1
''')

(DBT / "models/marts/dim_customer_scd2.sql").write_text('''
select customer_id, name, city, segment,
       cast(dbt_valid_from as date) as effective_from,
       cast(coalesce(dbt_valid_to, timestamp '9999-12-31') as date) as effective_to,
       dbt_valid_to is null as is_current
from {{ ref('customers_snapshot') }}
''')

(DBT / "models/marts/fct_orders.sql").write_text('''
{{ config(materialized='incremental', unique_key='order_id') }}
select o.order_id, o.customer_id, o.order_date, o.amount, o.currency, d.city as customer_city_at_order, d.segment as segment_at_order
from {{ ref('stg_orders') }} o
left join {{ ref('dim_customer_scd2') }} d
  on o.customer_id = d.customer_id and o.order_date >= d.effective_from and o.order_date < d.effective_to
{% if is_incremental() %}
where o.order_date >= (select coalesce(max(order_date), date '1900-01-01') from {{ this }})
{% endif %}
''')

(DBT / "models/schema.yml").write_text('''
version: 2
models:
  - name: stg_orders
    columns:
      - name: order_id
        tests: [unique, not_null]
      - name: currency
        tests:
          - accepted_values: {values: ['GBP', 'USD', 'MUR']}
  - name: dim_customer_scd2
    columns:
      - name: customer_id
        tests: [not_null]
  - name: fct_orders
    columns:
      - name: order_id
        tests: [unique, not_null]
      - name: customer_id
        tests:
          - relationships: {to: ref('dim_customer_scd2'), field: customer_id}
''')

(DBT / "tests/assert_one_current_row_per_customer.sql").write_text('''
select customer_id, count(*) as n
from {{ ref('dim_customer_scd2') }}
where is_current
group by 1
having count(*) > 1
''')
print(*sorted(str(p.relative_to(DBT)) for p in DBT.rglob("*") if p.is_file()), sep="\n")

In [ ]:
RUN_DBT = False     # set True after: pip install dbt-duckdb
import subprocess
def dbt(*args):
    r = subprocess.run(["dbt", *args, "--profiles-dir", "."], cwd=DBT, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-800:]); return r.returncode

if RUN_DBT:
    dbt("seed"); dbt("snapshot")                                   # day 1 history
    shutil.copy(DATA / "customers_day2.csv", DBT / "seeds/customers.csv")
    dbt("seed"); dbt("snapshot")                                   # day 2 -> dbt closes old rows, opens new ones
    dbt("build")                                                   # models + tests
else:
    print("dbt run skipped. Commands:  cd pipeline_lab/dbt_project && dbt seed && dbt snapshot && dbt build --profiles-dir .")

**Spark vs dbt SCD2: when to use which**

| | PySpark SCD2 | dbt snapshot |
|---|---|---|
| Strength | Full control, huge volumes, streaming/complex logic | Declarative, tested, documented, lineage graph for free |
| Weakness | You own the edge cases | Snapshot runs must be scheduled regularly; limited custom logic |
| Typical use | Lakehouse (Databricks/EMR) | Warehouse (Snowflake/BigQuery/Redshift) |

Most UK/US job descriptions want **both mindsets**: Spark for heavy lifting, dbt for modelling and tests.

In [ ]:
spark.stop()